# SHIVA-LLM · Objective 02 · Stage 6
## Frozen-policy ablations, stress tests, and statistical analysis

**Colab runtime:** Python 3, CPU; 12–16 GB RAM recommended.

**Upload:** `objective02_stage4a_simulator.zip`, `objective02_stage4ac_reward_redesign.zip`, `objective02_stage4b_policy_optimization.zip`, and `objective02_stage5_comparative_evaluation.zip`.

**Scope:** One-step, assumption-based contextual bandit. We never retrain or select a policy using test data. All Stage 6 test analyses are **post-hoc exploratory** and should not be used to tune Stage 4B. No real causal instructional effects or longitudinal learning improvements are established.

**Ablation types:** (A) reward-component removal (changes the evaluation objective), (B) frozen-policy readiness perturbation (tests input sensitivity), (C) assumed action-effect scenarios, and (D) simple comparator rules. Learner-cluster paired bootstraps estimate sampling uncertainty *conditional on the assumed simulator*.

In [ ]:
import os, io, json, zipfile, pathlib, hashlib
import numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib
from sklearn.metrics import roc_auc_score, brier_score_loss
from google.colab import files
SEED=42
OUT=pathlib.Path('/content/stage6_results'); OUT.mkdir(parents=True,exist_ok=True)
INPUTS=['objective02_stage4a_simulator.zip','objective02_stage4ac_reward_redesign.zip',
        'objective02_stage4b_policy_optimization.zip','objective02_stage5_comparative_evaluation.zip']
uploaded=files.upload()
missing=[name for name in INPUTS if not os.path.isfile(name)]
assert not missing, f'Missing ZIPs: {missing}'
Z=[zipfile.ZipFile(name) for name in INPUTS]
assert all(z.testzip() is None for z in Z),'Corrupt input ZIP'
for name,z in zip(INPUTS,Z): print(name,':',z.namelist())

In [ ]:
p=pd.read_csv(io.BytesIO(Z[0].read('response_model_predictions.csv')))
redesign=json.loads(Z[1].read('stage4ac_report.json'))
b_report=json.loads(Z[2].read('stage4b_report.json'))
s5=json.loads(Z[3].read('stage5_report.json'))
s5_compare=pd.read_csv(io.BytesIO(Z[3].read('replication_policy_comparison.csv')))
model=joblib.load(io.BytesIO(Z[2].read('stage4b_selected_policy.joblib')))
assert {'split','user_id','order_id','observed_correct','p_observed'} <= set(p)
assert set(p.split.unique())=={'train','val','test'}
assert not p.duplicated(['user_id','order_id']).any()
assert p.p_observed.between(0,1).all()
users={s:set(p.loc[p.split==s,'user_id']) for s in ['train','val','test']}
assert not users['train']&users['val'] and not users['train']&users['test'] and not users['val']&users['test']
assert b_report['selected_policy']=='tree_depth_2'
assert b_report['train_only_fit'] and b_report['validation_only_model_selection'] and b_report['test_not_used_for_selection']
assert s5['stage4b_replication_pass'] and s5['replication_max_absolute_reward_difference']<1e-8
assert b_report['frozen_stage4ac_reward']==redesign['selected_design']
assert model.n_features_in_==1
test=p[p.split=='test'].reset_index(drop=True)
assert len(test)==32522 and test.user_id.nunique()==402
P=test.p_observed.to_numpy(dtype=float)
Y=test.observed_correct.to_numpy(dtype=int)
print('Verified',len(test),'test events,',test.user_id.nunique(),'test learners')

### Frozen simulator and policies

Reward components: expected correctness, assumed challenge credit, assumed suitability, and assumed cost. Challenge credit and cost are *not* observed learning effects. When removing a reward component, **the reward objective changes**; therefore cross-ablation reward values are not directly comparable.

In [ ]:
W=dict(b_report['reward_parameters'])
E=dict(b_report['effect_scenarios'])
CH=np.asarray(b_report['challenge_credits'],float)
COST=np.asarray(b_report['assumed_costs'],float)
ACTIONS=['easy','medium','hard']
def sigmoid(x): return 1/(1+np.exp(-np.clip(x,-35,35)))
def outcomes(prob,d):
    prob=np.clip(np.asarray(prob,dtype=float),1e-6,1-1e-6)
    l=np.log(prob/(1-prob))
    return sigmoid(l[:,None]+np.array([d,0,-d])[None,:])
def components(prob,d):
    q=outcomes(prob,d)
    return {'correctness':q,
            'challenge':q*CH[None,:],
            'suitability':np.exp(-.5*((q-W['tau'])/W['h'])**2),
            'cost':np.broadcast_to(COST[None,:],q.shape)}
def utility(prob,d,weights=None):
    w=W if weights is None else weights
    c=components(prob,d)
    return (w['wc']*c['correctness']+w['wm']*c['challenge']
            +w['ws']*c['suitability']-w['wk']*c['cost'])
LEARNED=np.asarray(model.predict(P.reshape(-1,1)),dtype=int)
assert np.isin(LEARNED,[0,1,2]).all()
RULE=np.where(P<.4,0,np.where(P<.7,1,2)).astype(int)
BASELINES={'learned':LEARNED,'readiness_rule':RULE,
 'fixed_easy':np.zeros(len(P),dtype=int),
 'fixed_medium':np.ones(len(P),dtype=int),
 'fixed_hard':np.full(len(P),2,dtype=int),
 'random_seed42':np.random.default_rng(SEED).integers(0,3,len(P))}
def mean_reward(U,actions):return float(U[np.arange(len(U)),actions].mean())
def metrics(U,actions):
    return {'mean_reward':mean_reward(U,actions),
            'mean_regret':float(np.mean(U.max(axis=1)-U[np.arange(len(U)),actions])),
            **{f'fraction_{a}':float(np.mean(actions==i)) for i,a in enumerate(ACTIONS)}}

In [ ]:
# Check original Stage 5 rewards to prevent silent implementation drift.
rep=[]
for scenario,d in E.items():
    U=utility(P,d)
    for name,acts in BASELINES.items():
        original_name='random' if name=='random_seed42' else name
        ref=s5_compare[(s5_compare.scenario==scenario)&(s5_compare.policy==original_name)]
        assert len(ref)==1,(scenario,original_name)
        value=mean_reward(U,acts)
        expected=float(ref.iloc[0].mean_simulated_reward)
        rep.append({'scenario':scenario,'policy':name,'computed':value,'stage5':expected,
                    'absolute_difference':abs(value-expected)})
rep=pd.DataFrame(rep)
display(rep)
assert rep.absolute_difference.max()<1e-8, 'Stage 5 replication mismatch: stop and inspect'
rep.to_csv(OUT/'stage5_replication_check.csv',index=False)

### A. Leave-one-component-out reward ablation

For each ablation, the removed coefficient is set to zero and the **remaining positive coefficients are renormalized** to sum to 1. The cost remains a penalty. We also include the unmodified objective. These are counterfactual **preferences**, not empirical intervention effects.

In [ ]:
WEIGHT_KEYS={'correctness':'wc','challenge':'wm','suitability':'ws','cost':'wk'}
def without_component(component):
    w=W.copy()
    w[WEIGHT_KEYS[component]]=0.
    total=sum(w[k] for k in ['wc','wm','ws','wk'])
    assert total>0
    for k in ['wc','wm','ws','wk']:w[k]/=total
    return w
ablation_weights={'full':W,**{f'no_{name}':without_component(name) for name in WEIGHT_KEYS}}
ab_rows=[]
for ab,w in ablation_weights.items():
    for scenario,d in E.items():
        U=utility(P,d,w)
        oracle=U.argmax(axis=1)
        best_fixed=max(mean_reward(U,BASELINES[k]) for k in ['fixed_easy','fixed_medium','fixed_hard'])
        for policy,act in {**BASELINES,'assumed_oracle':oracle}.items():
            ab_rows.append({'ablation':ab,'scenario':scenario,'policy':policy,
                            **metrics(U,act),'best_fixed_reward':best_fixed,
                            'advantage_over_best_fixed':mean_reward(U,act)-best_fixed})
ab=pd.DataFrame(ab_rows)
ab.to_csv(OUT/'reward_component_ablation.csv',index=False)
display(ab[(ab.scenario=='moderate')&ab.policy.isin(['learned','readiness_rule','assumed_oracle'])][
 ['ablation','policy','mean_reward','advantage_over_best_fixed','mean_regret']])

### B. Input-feature perturbation

The frozen policy only accepts **one feature**, `p_observed`. We therefore perturb that input, not the original multi-feature HyperKT architecture. The calibrated Stage 4A model is not retrained. Test-set perturbations are exploratory stress tests, not a new clean holdout.

For each perturbation, evaluate actions selected using perturbed readiness against the **original unperturbed simulated utility**. This isolates policy sensitivity rather than redefining ground-truth rewards.

In [ ]:
def perturb(prob,kind,seed):
    rg=np.random.default_rng(seed)
    if kind=='none':return prob.copy()
    if kind=='gaussian_0.025':return np.clip(prob+rg.normal(0,.025,len(prob)),1e-6,1-1e-6)
    if kind=='gaussian_0.05':return np.clip(prob+rg.normal(0,.05,len(prob)),1e-6,1-1e-6)
    if kind=='gaussian_0.10':return np.clip(prob+rg.normal(0,.10,len(prob)),1e-6,1-1e-6)
    if kind=='bias_plus_0.05':return np.clip(prob+.05,1e-6,1-1e-6)
    if kind=='bias_minus_0.05':return np.clip(prob-.05,1e-6,1-1e-6)
    if kind=='shuffle_within_learner':
        out=prob.copy()
        for idx in test.groupby('user_id').indices.values():
            out[idx]=rg.permutation(out[idx])
        return out
    raise ValueError(kind)
KINDS=['none','gaussian_0.025','gaussian_0.05','gaussian_0.10',
       'bias_plus_0.05','bias_minus_0.05','shuffle_within_learner']
U0=utility(P,E['moderate'])
stress=[]
for kind in KINDS:
    for seed in range(10):
        pert=perturb(P,kind,SEED+seed)
        a=np.asarray(model.predict(pert.reshape(-1,1)),int)
        stress.append({'perturbation':kind,'seed':seed,
            'mean_reward_on_original_context':mean_reward(U0,a),
            'regret_on_original_context':float(np.mean(U0.max(axis=1)-U0[np.arange(len(P)),a])),
            'action_change_fraction':float(np.mean(a!=LEARNED)),
            'mean_absolute_probability_change':float(np.mean(np.abs(pert-P)))})
stress=pd.DataFrame(stress)
stress.to_csv(OUT/'readiness_perturbation_stress.csv',index=False)
display(stress.groupby('perturbation').agg(
 reward_mean=('mean_reward_on_original_context','mean'),
 reward_std=('mean_reward_on_original_context','std'),
 changed=('action_change_fraction','mean')).reset_index())

### C. Paired learner-level uncertainty for reward-component ablations

Report paired contrasts of the **frozen learned policy minus the readiness rule** within each modified objective, under the moderate assumed effect. Resampling is by learner; intervals do not represent uncertainty in the assumed reward mechanism.

In [ ]:
GROUPS=list(test.groupby('user_id').indices.values())
def paired_cluster_ci(diff,seed=42,B=500):
    rg=np.random.default_rng(seed)
    G=len(GROUPS)
    # per-learner sums and counts avoid repeated large array concatenations
    sums=np.array([diff[idx].sum() for idx in GROUPS])
    counts=np.array([len(idx) for idx in GROUPS])
    draws=rg.integers(0,G,size=(B,G))
    boot=sums[draws].sum(axis=1)/counts[draws].sum(axis=1)
    return {'point_difference':float(diff.mean()),
            'ci_95_low':float(np.quantile(boot,.025)),
            'ci_95_high':float(np.quantile(boot,.975)),
            'bootstrap_reps':B}
ci_rows=[]
for j,(name,w) in enumerate(ablation_weights.items()):
    U=utility(P,E['moderate'],w)
    diff=U[np.arange(len(P)),LEARNED]-U[np.arange(len(P)),RULE]
    ci_rows.append({'ablation':name,**paired_cluster_ci(diff,SEED+j)})
ci=pd.DataFrame(ci_rows)
display(ci)
ci.to_csv(OUT/'paired_learner_ablation_ci.csv',index=False)

### D. Readiness calibration and subgroup diagnostics

Calibration is evaluated against **observed correctness**, but simulated action rewards are not. This analysis is descriptive and uses the existing locked test data.

In [ ]:
bins=[0,.4,.7,1.000001]
labels=['low','medium','high']
groups=pd.cut(P,bins=bins,labels=labels,include_lowest=True,right=False)
cal=[]
for label in labels:
    mask=np.asarray(groups==label)
    yy=Y[mask];pp=P[mask]
    cal.append({'group':label,'events':int(mask.sum()),
       'learners':int(test.loc[mask,'user_id'].nunique()),
       'observed_correct':float(yy.mean()),'predicted_correct':float(pp.mean()),
       'calibration_gap_pred_minus_observed':float(pp.mean()-yy.mean()),
       'brier':float(brier_score_loss(yy,pp)),
       'roc_auc':float(roc_auc_score(yy,pp)) if len(np.unique(yy))==2 else np.nan})
cal=pd.DataFrame(cal)
cal.to_csv(OUT/'readiness_calibration.csv',index=False)
display(cal)

In [ ]:
# Plots and summary
fig,axes=plt.subplots(1,2,figsize=(12,4))
mod=ab[(ab.scenario=='moderate')&(ab.policy=='learned')]
axes[0].bar(mod.ablation,mod.advantage_over_best_fixed)
axes[0].axhline(0,linewidth=1)
axes[0].tick_params(axis='x',rotation=45)
axes[0].set_title('Reward-component ablations')
axes[0].set_ylabel('Learned minus best fixed (within each objective)')
ss=stress.groupby('perturbation').action_change_fraction.mean().reindex(KINDS)
axes[1].bar(range(len(ss)),ss.values)
axes[1].set_xticks(range(len(ss)),ss.index,rotation=60,ha='right')
axes[1].set_title('Frozen-policy action sensitivity')
axes[1].set_ylabel('Fraction of actions changed')
fig.tight_layout()
fig.savefig(OUT/'stage6_ablation_figures.png',dpi=170)
plt.show()

In [ ]:
# Machine-readable report with limitations.
report={
 'stage':'6','experiment':'Frozen-policy simulated ablations',
 'test_events':int(len(test)),'test_learners':int(test.user_id.nunique()),
 'selected_model':b_report['selected_policy'],
 'stage5_replication_max_error':float(rep.absolute_difference.max()),
 'original_reward':W,'effect_scenarios':E,
 'reward_ablation_weights':ablation_weights,
 'readiness_perturbations':KINDS,
 'policy_retrained':False,'test_used_for_selection':False,
 'analyses_are_posthoc_exploratory':True,
 'observed_causal_action_effects':False,
 'validated_longitudinal_RL':False,
 'real_world_learning_gain_established':False,
 'limitations':[
  'Reward ablations change the objective and must not be compared as identical utility scales.',
  'Readiness perturbations act on the single frozen-policy input, not directly on the HyperKT model.',
  'Test-set stress analyses are exploratory and cannot be used to choose a revised policy.',
  'Paired bootstrap CIs condition on synthetic action effects and reward assumptions.',
  'Synthetic challenge credit is not observed mastery improvement.']}
(OUT/'stage6_report.json').write_text(json.dumps(report,indent=2))
(OUT/'README.txt').write_text(
 'SHIVA-LLM Objective 02 Stage 6: frozen-policy simulation ablations.\n'
 'No empirical action-effect identification or observed learning gains.\n'
 'Inputs: Stage 4A, Stage 4A-C, Stage 4B, Stage 5 ZIPs.\n')
zip_path=pathlib.Path('/content/objective02_stage6_ablation_analysis.zip')
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for f in sorted(OUT.iterdir()):
        if f.is_file():z.write(f,arcname=f.name)
print('Saved',zip_path,'bytes',zip_path.stat().st_size)
files.download(str(zip_path))